# Veritas — Colab GPU training (no CIFAKE)

**Does everything from scratch on GPU:** downloads COCO reals + newer-generator fakes (MJ/nano/art) + 3x AUTO sets, builds a balanced `FINAL` set, trains EfficientNet-B0 @384px, exports ONNX.

**Steps:**
1. `Runtime > Change runtime type > T4 GPU` (required)
2. `Runtime > Run all`, paste your `HF_TOKEN` when asked
3. At the end, two files download: `hq_efficientnet_b0.pt` + `.onnx` → copy both into your local `weights\` folder

CIFAKE is intentionally excluded (32px, single-generator). All training data here is high-res multi-generator.

## 0. Config (edit quotas here if you want more/less data)

In [ ]:
CONFIG = {
    # AUTO streaming sets (split EVENLY across the 3 sources below)
    "AUTO_TRAIN_PER_CLASS": 2500,
    "AUTO_TEST_PER_CLASS": 500,
    # FINAL merged set (AUTO + COCO + MJ/nano/art)
    "FINAL_TRAIN_PER_CLASS": 4000,
    "FINAL_TEST_PER_CLASS": 1000,
    # newer-fake extraction caps (train)
    "MJ_TRAIN": 400, "NANO_TRAIN": 700, "ART_TRAIN": 400,
    "MJ_TEST": 100, "NANO_TEST": 175, "ART_TEST": 100,
    # training
    "IMG_SIZE": 384, "EPOCHS": 12, "BATCH": 64, "LR": 3e-4, "PATIENCE": 4,
}
AUTO_SOURCES = [
    "Shanmuk4622/ai-image-detection-dataset",  # fallback to -v3 tried automatically
    "Shanmuk4622/ai-detection-dataset-v3",
    "Rajarshi-Roy-research/Defactify_Image_Dataset",
    "TheKernel01/Tiny-GenImage",
]  # dupes by content-hash are skipped, so overlap is harmless
print(CONFIG)

## 1. GPU check + installs

In [ ]:
!nvidia-smi -L
!pip install -q datasets huggingface_hub pillow tqdm pandas pyarrow onnx onnxruntime
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "STOP: set Runtime > Change runtime type > T4 GPU, then re-run"
print("GPU:", torch.cuda.get_device_name(0))

## 2. HF login (paste token — needed for gated sets)

In [ ]:
import os
from getpass import getpass
tok = os.getenv("HF_TOKEN", "") or getpass("HF_TOKEN (hf_...): ")
os.environ["HF_TOKEN"] = tok.strip()
from huggingface_hub import login, whoami
login(token=os.environ["HF_TOKEN"])
print("logged in as:", whoami()["name"])

## 3. Download COCO reals (~900MB) + newer-generator raws (~1.2GB)

In [ ]:
import os, time, urllib.request, zipfile
t0 = time.perf_counter()
os.makedirs("/content/data/_raw", exist_ok=True)
os.makedirs("/content/data/HQ_SRC_REAL", exist_ok=True)

# --- COCO val2017 reals (same source as local dl_hq_real.py) ---
URL = "http://images.cocodataset.org/zips/val2017.zip"
TMP = "/content/data/_raw/val2017.zip"
if not os.path.isfile(TMP):
    print("COCO val2017 (~900MB) downloading...", flush=True)
    urllib.request.urlretrieve(URL, TMP)
    print(f"COCO zip done in {(time.perf_counter()-t0)/60:.1f} min", flush=True)
with zipfile.ZipFile(TMP) as z:
    for m in z.infolist():
        if not m.filename.lower().endswith(".jpg"):
            continue
        target = os.path.join("/content/data/HQ_SRC_REAL", "coco_" + os.path.basename(m.filename))
        if os.path.isfile(target):
            continue
        with z.open(m) as src, open(target, "wb") as dst:
            dst.write(src.read())
n_coco = len([f for f in os.listdir("/content/data/HQ_SRC_REAL") if f.endswith(".jpg")])
print(f"COCO reals: {n_coco}  [t+{(time.perf_counter()-t0)/60:.1f} min]")

# --- MJ + nano parquet shards + art imagefolder (same IDs as local dl_newer.py) ---
from huggingface_hub import hf_hub_download, snapshot_download
mj = hf_hub_download("bitmind/GenImage_MidJourney", filename="data/train-00000-of-00402.parquet",
                     repo_type="dataset", local_dir="/content/data/_raw/mj")
print("MJ shard:", mj, flush=True)
nb = hf_hub_download("julienlucas/midjourney-dalle-sd-nanobananapro-dataset",
                     filename="data/train-00000-of-00009.parquet",
                     repo_type="dataset", local_dir="/content/data/_raw/nano")
print("nano shard:", nb, flush=True)
art = snapshot_download("hmnshudhmn24/real-fake-ai-generated-art-images", repo_type="dataset",
                        local_dir="/content/data/_raw/art", allow_patterns=["Data/*"])
print("art:", art, flush=True)
print(f"ALL RAW DONE [t+{(time.perf_counter()-t0)/60:.1f} min]")

## 4. Stream the 3 AUTO sets → `/content/data/AUTO` (512px Q95, SHA256 dedup)

In [ ]:
import glob, hashlib, io, random, time
from PIL import Image, ImageOps
from datasets import load_dataset

SEED = 7
rng = random.Random(SEED)
OUT = "/content/data/AUTO"
for sp in ("train", "test"):
    for cl in ("REAL", "FAKE"):
        os.makedirs(f"{OUT}/{sp}/{cl}", exist_ok=True)

n_src = 3
q_tr = CONFIG["AUTO_TRAIN_PER_CLASS"] // n_src
q_te = CONFIG["AUTO_TEST_PER_CLASS"] // n_src
test_ratio = q_te / max(1, q_tr + q_te)
print(f"quota/source: train {q_tr}/class, test {q_te}/class")

def canon(im):
    im = ImageOps.exif_transpose(im).convert("RGB")
    w, h = im.size
    s = min(w, h)
    im = im.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2)).resize((512,512), Image.LANCZOS)
    b = io.BytesIO(); im.save(b, "JPEG", quality=95)
    return b.getvalue()

def to_pil(o):
    try:
        if isinstance(o, Image.Image): return o
        if isinstance(o, dict) and o.get("bytes"):
            return Image.open(io.BytesIO(o["bytes"]))
        if isinstance(o, (bytes, bytearray)):
            return Image.open(io.BytesIO(bytes(o)))
    except Exception:
        return None

def to_label(o):
    if isinstance(o, bool): return int(o)
    if isinstance(o, (int, float)) and int(o) in (0,1): return int(o)
    s = str(o).strip().lower()
    if s in ("real","human","natural","0"): return 0
    if s in ("fake","ai","generated","synthetic","1"): return 1

def try_load(ds_id):
    try:
        ds = load_dataset(ds_id, split="train", streaming=True, token=os.environ["HF_TOKEN"])
        return ds, list(ds.features.keys())
    except Exception as e:
        return None, [f"ERR: {e}"]

# resolve Shanmuk ID (user ID first, -v3 fallback), then fixed trio
SOURCES, seen = [], set()
for cand in AUTO_SOURCES:
    ds, cols = try_load(cand)
    if ds is None:
        print(f"[SKIP] {cand}: {cols[0][:160]}")
        continue
    key = cand.lower()
    # avoid double-counting same repo under two names: check dataset id fingerprint via first cols
    if key in seen: continue
    seen.add(key)
    SOURCES.append((cand, ds, cols))
    print(f"[OK] {cand} cols={cols}")
    if len(SOURCES) == 3: break
assert len(SOURCES) >= 2, "Need >=2 AUTO sources (check token + gated approval)"
# re-split quota over ACTUAL source count
n_src = len(SOURCES)
q_tr = CONFIG["AUTO_TRAIN_PER_CLASS"] * 3 // n_src // 3 * 3  # keep simple: equal split
q_tr = CONFIG["AUTO_TRAIN_PER_CLASS"] // n_src
q_te = CONFIG["AUTO_TEST_PER_CLASS"] // n_src
print(f"using {n_src} sources, quota/source train {q_tr}/class test {q_te}/class")

t_all = time.perf_counter()
exist = set()
for p in glob.glob(f"{OUT}/*/*/auto_*"):
    with open(p,"rb") as f: exist.add(hashlib.sha256(f.read()).hexdigest())
print(f"[resume] {len(exist)} existing hashes")

for ds_id, ds, cols in SOURCES:
    tag = ds_id.split("/")[-1].lower().replace("-","")[:6]
    low = {c.lower(): c for c in cols}
    ic = next((low[c] for c in ("image","img") if c in low), cols[0])
    lc = next((low[c] for c in ("label","label_a") if c in low), None)
    if lc is None:
        print(f"[SKIP] {ds_id}: no label col in {cols}"); continue
    got = {"train":{"REAL":0,"FAKE":0}, "test":{"REAL":0,"FAKE":0}}
    # resume offsets from filenames
    for sp in ("train","test"):
        for cl in ("REAL","FAKE"):
            got[sp][cl] = len(glob.glob(f"{OUT}/{sp}/{cl}/auto_{tag}_*.jpg"))
    scanned = kept = dup = 0; t_ds = time.perf_counter()
    for row in ds:
        scanned += 1
        if scanned > 60000: break
        y = to_label(row.get(lc))
        if y is None: continue
        cl = "FAKE" if y==1 else "REAL"
        sp = "test" if rng.random() < test_ratio else "train"
        q = q_te if sp=="test" else q_tr
        if got[sp][cl] >= q:
            alt = "train" if sp=="test" else "test"
            aq = q_tr if alt=="train" else q_te
            if got[alt][cl] >= aq: continue
            sp = alt
        im = to_pil(row.get(ic))
        if im is None or min(im.size) < 224: continue
        try: data = canon(im)
        except Exception: continue
        h = hashlib.sha256(data).hexdigest()
        if h in exist: dup += 1; continue
        with open(f"{OUT}/{sp}/{cl}/auto_{tag}_{got[sp][cl]:05d}.jpg","wb") as f: f.write(data)
        exist.add(h); got[sp][cl]+=1; kept+=1
        if kept % 200 == 0:
            el=time.perf_counter()-t_ds
            print(f"[{tag}] kept={kept} scanned={scanned} dup={dup} {kept/max(el,1e-6):.1f}/s "
                  f"tr R/F={got['train']['REAL']}/{got['train']['FAKE']} "
                  f"te R/F={got['test']['REAL']}/{got['test']['FAKE']}", flush=True)
        if all([got["train"]["REAL"]>=q_tr, got["train"]["FAKE"]>=q_tr,
                got["test"]["REAL"]>=q_te, got["test"]["FAKE"]>=q_te]): break
    print(f"[{tag}] DONE kept={kept} dup={dup} scanned={scanned} "
          f"tr R/F={got['train']['REAL']}/{got['train']['FAKE']} "
          f"te R/F={got['test']['REAL']}/{got['test']['FAKE']}")
print(f"AUTO DONE [t+{(time.perf_counter()-t_all)/60:.1f} min]")

## 5. Build balanced `FINAL` (AUTO + COCO + MJ/nano/art — no CIFAKE)

In [ ]:
import glob, io, random, shutil, time
import pandas as pd
from PIL import Image, ImageOps

t0=time.perf_counter(); rng=random.Random(7)
FINAL="/content/data/FINAL"
shutil.rmtree(FINAL, ignore_errors=True)
for sp in ("train","test"):
    for cl in ("REAL","FAKE"): os.makedirs(f"{FINAL}/{sp}/{cl}", exist_ok=True)

def canon_save(src_img, dst):
    im = ImageOps.exif_transpose(src_img).convert("RGB")
    w,h = im.size; s=min(w,h)
    im = im.crop(((w-s)//2,(h-s)//2,(w+s)//2,(h+s)//2)).resize((512,512), Image.LANCZOS)
    im.save(dst,"JPEG",quality=95)

# --- newer fakes from parquet (caps from CONFIG) ---
def parquet_fakes(pq, dest, prefix, take, skip=0):
    df = pd.read_parquet(pq, columns=["image"])
    n=0
    for i in range(skip, min(skip+take*3, len(df))):  # overscan (size filter drops some)
        if n>=take: break
        c = df["image"].iloc[i]
        raw = c["bytes"] if isinstance(c,dict) else c
        try:
            im = Image.open(io.BytesIO(raw))
            if min(im.size)<224: continue
            im.load()
            canon_save(im, f"{dest}/{prefix}_{n:05d}.jpg"); n+=1
        except Exception: continue
    return n
TMPF="/content/data/_tmpfake"; shutil.rmtree(TMPF, ignore_errors=True)
os.makedirs(TMPF, exist_ok=True)
n_mj = parquet_fakes("/content/data/_raw/mj/data/train-00000-of-00402.parquet", TMPF, "mj",
                     CONFIG["MJ_TRAIN"]+CONFIG["MJ_TEST"])
n_nb = parquet_fakes("/content/data/_raw/nano/data/train-00000-of-00009.parquet", TMPF, "nano",
                     CONFIG["NANO_TRAIN"]+CONFIG["NANO_TEST"])
art_src="/content/data/_raw/art/Data/FAKE"
art_all=sorted(glob.glob(f"{art_src}/*")); rng.shuffle(art_all)
n_art=0
for p in art_all:
    if n_art >= CONFIG["ART_TRAIN"]+CONFIG["ART_TEST"]: break
    try:
        with Image.open(p) as im:
            if min(im.size)<224: continue
            im.load(); canon_save(im, f"{TMPF}/art_{n_art:05d}.jpg"); n_art+=1
    except Exception: continue
print(f"newer fakes: mj={n_mj} nano={n_nb} art={n_art}", flush=True)

# --- pools ---
auto_tr_R=sorted(glob.glob("/content/data/AUTO/train/REAL/*.jpg"))
auto_tr_F=sorted(glob.glob("/content/data/AUTO/train/FAKE/*.jpg"))
auto_te_R=sorted(glob.glob("/content/data/AUTO/test/REAL/*.jpg"))
auto_te_F=sorted(glob.glob("/content/data/AUTO/test/FAKE/*.jpg"))
def _big_enough(p, px=448):
    try:
        with Image.open(p) as im: return min(im.size) >= px
    except Exception: return False
coco=[p for p in sorted(glob.glob("/content/data/HQ_SRC_REAL/*.jpg")) if _big_enough(p)][:5000]
newer=sorted(glob.glob(f"{TMPF}/*.jpg"))
print(f"pools: auto tr R/F={len(auto_tr_R)}/{len(auto_tr_F)} te R/F={len(auto_te_R)}/{len(auto_te_F)} | coco={len(coco)} | newer={len(newer)}")

def fill(dest_sp, dest_cl, srcs, need, tag):
    rng.shuffle(srcs); n=0
    for s in srcs:
        if n>=need: break
        try:
            with Image.open(s) as im:
                im.load(); canon_save(im, f"{FINAL}/{dest_sp}/{dest_cl}/{tag}_{n:05d}.jpg"); n+=1
        except Exception: continue
    return n
TR, TE = CONFIG["FINAL_TRAIN_PER_CLASS"], CONFIG["FINAL_TEST_PER_CLASS"]
# REAL: AUTO + COCO split ~50/50
r_tr = fill("train","REAL", auto_tr_R[:TR]+coco[:TR], TR, "real")
r_te = fill("test","REAL", auto_te_R[:TE]+coco[TR:TR+TE], TE, "real")
# FAKE: AUTO + newer split
f_tr = fill("train","FAKE", auto_tr_F[:TR]+newer[:TR], TR, "fake")
f_te = fill("test","FAKE", auto_te_F[:TE]+newer[TR:TR+TE], TE, "fake")
# balance REAL==FAKE per split
for sp,a,b in [("train",r_tr,f_tr),("test",r_te,f_te)]:
    m=min(a,b)
    for cl in ("REAL","FAKE"):
        for p in sorted(glob.glob(f"{FINAL}/{sp}/{cl}/*.jpg"))[m:]: os.remove(p)
    print(f"{sp}: REAL==FAKE=={m}")
print(f"FINAL DONE [t+{(time.perf_counter()-t0)/60:.1f} min]")

## 6. Train EfficientNet-B0 @384 on GPU (same head/transforms/checkpoint as local `train.py`)

In [ ]:
import time, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models
from torchvision.transforms import v2 as T
from tqdm import tqdm

IM=CONFIG["IMG_SIZE"]; device=torch.device("cuda")
mean, std = [0.485,0.456,0.406],[0.229,0.224,0.225]
train_tf=T.Compose([T.RandomResizedCrop(IM,scale=(0.7,1.0)),T.RandomHorizontalFlip(),
  T.ColorJitter(0.1,0.1,0.1,0.05),T.GaussianBlur(3,(0.1,1.0)),T.ToImage(),
  T.ToDtype(torch.float32,scale=True),T.Normalize(mean, std)])
eval_tf=T.Compose([T.Resize(int(IM*1.15)),T.CenterCrop(IM),T.ToImage(),
  T.ToDtype(torch.float32,scale=True),T.Normalize(mean, std)])
def flip(i): return 1-int(i)  # ImageFolder FAKE=0,REAL=1 -> P(FAKE)
tr_ds=datasets.ImageFolder("/content/data/FINAL/train",transform=train_tf,target_transform=flip)
te_ds=datasets.ImageFolder("/content/data/FINAL/test",transform=eval_tf,target_transform=flip)
print("classes:",tr_ds.classes,"| train:",len(tr_ds),"| test:",len(te_ds))
tr_ld=DataLoader(tr_ds,batch_size=CONFIG["BATCH"],shuffle=True,num_workers=4,pin_memory=True,persistent_workers=True)
te_ld=DataLoader(te_ds,batch_size=CONFIG["BATCH"]*2,shuffle=False,num_workers=4,pin_memory=True)

net=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
net.classifier[1]=nn.Linear(net.classifier[1].in_features,1)
net.to(device, memory_format=torch.channels_last)
opt=torch.optim.AdamW(net.parameters(),lr=CONFIG["LR"],weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CONFIG["EPOCHS"])
crit=nn.BCEWithLogitsLoss(); scaler=torch.amp.GradScaler("cuda")

@torch.no_grad()
def ev():
    net.eval(); l=c=t=0
    for x,y in te_ld:
        x=x.to(device,memory_format=torch.channels_last); y=y.float().to(device).unsqueeze(1)
        lg=net(x); l+=crit(lg,y).item()*len(x); c+=((lg.sigmoid()>=0.5).float()==y).sum().item(); t+=len(x)
    return l/t, c/t

best, bad = 0.0, 0
for ep in range(1, CONFIG["EPOCHS"]+1):
    net.train(); run=0; t0=time.perf_counter()
    pb=tqdm(tr_ld,desc=f"epoch {ep}/{CONFIG['EPOCHS']}",unit="batch")
    for x,y in pb:
        x=x.to(device,memory_format=torch.channels_last); y=y.float().to(device).unsqueeze(1)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            loss=crit(net(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run+=loss.item()*len(x); pb.set_postfix(loss=f"{loss.item():.4f}")
    sched.step(); vl,va=ev(); dt=time.perf_counter()-t0
    print(f"epoch {ep}: train_loss={run/len(tr_ds):.4f} val_loss={vl:.4f} val_acc={va:.4f} ({dt:.0f}s, {len(tr_ds)/dt:.0f} img/s)")
    if va>best:
        best,bad=va,0
        torch.save({"backbone":"efficientnet_b0","acc":va,"img_size":IM,"state_dict":net.state_dict()},"/content/hq_efficientnet_b0.pt")
        print(f"  * new best -> saved /content/hq_efficientnet_b0.pt (acc={va:.4f})")
    else:
        bad+=1
        if bad>=CONFIG["PATIENCE"]:
            print(f"early stopping ({CONFIG['PATIENCE']})"); break
print(f"\nBEST val_acc={best:.4f} -> /content/hq_efficientnet_b0.pt")

## 7. Export ONNX (same opset-18 recipe as local `export_onnx.py`) + download weights

In [ ]:
import numpy as np, os, torch, torch.nn as nn
from torchvision import models
ckpt=torch.load("/content/hq_efficientnet_b0.pt",map_location="cpu")
print("ckpt acc:",ckpt.get("acc"),"img:",ckpt.get("img_size"))
net=models.efficientnet_b0(weights=None)
net.classifier[1]=nn.Linear(net.classifier[1].in_features,1)
net.load_state_dict(ckpt["state_dict"],strict=False); net.eval()
IM=int(ckpt.get("img_size",384))
dummy=torch.randn(1,3,IM,IM)
with torch.no_grad(): ref=net(dummy).numpy()
torch.onnx.export(net,dummy,"/content/hq_efficientnet_b0.onnx",input_names=["input"],output_names=["logit"],
  dynamic_axes={"input":{0:"batch"},"logit":{0:"batch"}},opset_version=18,do_constant_folding=True,dynamo=False)
sz=os.path.getsize("/content/hq_efficientnet_b0.onnx")/1e6
print(f"onnx {sz:.1f} MB"); assert sz>5, "stub export, retry"
import onnxruntime as ort
sess=ort.InferenceSession("/content/hq_efficientnet_b0.onnx",providers=["CPUExecutionProvider"])
got=sess.run(None,{"input":dummy.numpy().astype("float32")})[0]
d=float(np.abs(got-ref).max()); same=bool((got.reshape(-1)[0]>=0)==(ref.reshape(-1)[0]>=0))
print(f"parity max|onnx-torch|={d:.2e} same_class={same}"); assert same and d<0.2
print("[OK] both weights ready in /content")
from google.colab import files
files.download("/content/hq_efficientnet_b0.pt")
files.download("/content/hq_efficientnet_b0.onnx")

## 8. Back on your PC

Copy both downloads into `weights\` (overwrite), then in CMD:
```cmd
set TORCH_WEIGHTS=weights\hq_efficientnet_b0.pt
set ONNX_WEIGHTS=weights\hq_efficientnet_b0.onnx
set IMAGE_SIZE=384
uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```